# QLoRA 4-bit Fine-Tuning

QLoRA combines a quantized frozen base model with trainable LoRA adapters. The base model is loaded in 4-bit precision to save memory, while gradients update only the adapter weights. This is why QLoRA made it practical to fine-tune larger LLMs on smaller GPUs.

## What You Will Learn

By the end, you should understand:

- how QLoRA differs from LoRA,
- what 4-bit base-model loading means,
- why NF4 is commonly used,
- what double quantization does,
- how bitsandbytes fits into the stack,
- how to prepare a model for k-bit training,
- how to configure QLoRA with PEFT,
- and how to avoid common memory traps.

## 1. LoRA vs QLoRA

| Method | Base model weights | Trainable weights | Main benefit |
|---|---|---|---|
| Full fine-tuning | fp16/bf16/fp32 | all weights | maximum flexibility |
| LoRA | fp16/bf16/fp32 | adapter weights | much cheaper training |
| QLoRA | 4-bit quantized | adapter weights | fine-tune larger models with less memory |

QLoRA does not train the 4-bit weights directly. It keeps the quantized base frozen and trains LoRA adapters. During computation, weights are dequantized as needed.

In [ ]:
import importlib.util

import pandas as pd

def has_package(name):
    return importlib.util.find_spec(name) is not None

packages = ["torch", "transformers", "peft", "bitsandbytes", "accelerate", "datasets"]
pd.DataFrame({"package": packages, "installed": [has_package(name) for name in packages]})

## 2. Install Notes

QLoRA typically needs GPU-compatible packages:

```bash
pip install transformers peft accelerate datasets bitsandbytes
```

`bitsandbytes` support depends on OS, CUDA, and hardware. If installation fails, study the configuration cells first and run QLoRA later in a suitable GPU environment.

## 3. Quantization Concepts

| Concept | Meaning | Why it matters |
|---|---|---|
| 4-bit loading | Store base weights with 4 bits per value | Reduces memory dramatically |
| NF4 | NormalFloat4 quantization type | Works well for normally distributed weights |
| Double quantization | Quantize quantization constants too | Saves additional memory |
| Compute dtype | dtype used during operations | bf16/fp16 affects speed and stability |
| Paged optimizer | Optimizer that reduces memory spikes | Helps avoid OOM during training |
| Gradient checkpointing | Recompute activations during backward pass | Saves memory at cost of extra compute |

In [ ]:
def rough_weight_memory_billions(parameters_b, bits_per_weight):
    bytes_per_weight = bits_per_weight / 8
    return parameters_b * 1_000_000_000 * bytes_per_weight / 1_000_000_000

rows = []
for params_b in [0.5, 1, 3, 7, 13]:
    rows.append({
        "model_size_b": params_b,
        "fp16_weights_gb": rough_weight_memory_billions(params_b, 16),
        "int8_weights_gb": rough_weight_memory_billions(params_b, 8),
        "int4_weights_gb": rough_weight_memory_billions(params_b, 4),
    })
pd.DataFrame(rows)

## 4. Build a BitsAndBytes Config

The central QLoRA object in Transformers is `BitsAndBytesConfig`. It tells the loader to use 4-bit quantization and selects the quantization type and compute dtype.

In [ ]:
try:
    import torch
    from transformers import BitsAndBytesConfig

    compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    print(bnb_config)
except Exception as error:
    bnb_config = None
    print("BitsAndBytesConfig skipped. Install transformers and bitsandbytes in a compatible environment.")
    print(f"Reason: {error}")

## 5. Load a Quantized Base Model

For a real QLoRA run, choose an instruction model that fits your GPU memory. This cell uses a small model name as a template. Keep it commented or guarded until your environment supports 4-bit loading.

In [ ]:
model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if not torch.cuda.is_available():
        raise RuntimeError("QLoRA 4-bit loading is best run in a CUDA GPU environment.")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
    print("loaded quantized model")
except Exception as error:
    tokenizer = None
    model = None
    print("Quantized model load skipped. Use a CUDA environment with bitsandbytes for this cell.")
    print(f"Reason: {error}")

## 6. Prepare for K-bit Training

PEFT provides `prepare_model_for_kbit_training`. It handles details needed when training adapters on top of a quantized model, such as input gradients and layer norm precision handling.

In [ ]:
try:
    from peft import prepare_model_for_kbit_training

    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()
    print("model prepared for k-bit training")
except Exception as error:
    print("K-bit preparation skipped. Load a quantized model first.")
    print(f"Reason: {error}")

## 7. Attach LoRA Adapters

QLoRA still uses LoRA adapters. The difference is the quantized frozen base. Target module names depend on the model family. Llama, Mistral, and Qwen-like models commonly use projection names such as `q_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, and `down_proj`.

In [ ]:
try:
    from peft import LoraConfig, TaskType, get_peft_model

    qlora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
    )
    model = get_peft_model(model, qlora_config)
    model.print_trainable_parameters()
except Exception as error:
    print("QLoRA adapter setup skipped. Target module names may need to match your model.")
    print(f"Reason: {error}")

## 8. Training Arguments for QLoRA

QLoRA training often uses small per-device batches, gradient accumulation, gradient checkpointing, and paged optimizers. Start conservative, then scale batch size and sequence length as memory allows.

In [ ]:
try:
    from transformers import TrainingArguments

    qlora_training_args = TrainingArguments(
        output_dir="finetuning_outputs/qlora",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        num_train_epochs=1,
        logging_steps=10,
        save_steps=100,
        eval_strategy="steps",
        eval_steps=100,
        optim="paged_adamw_8bit",
        fp16=False,
        bf16=False,
        report_to=[],
    )
    print(qlora_training_args)
except Exception as error:
    print("TrainingArguments setup skipped. Install transformers first.")
    print(f"Reason: {error}")

## 9. Memory Knobs

If QLoRA runs out of memory, tune these first:

| Knob | Lower memory direction | Trade-off |
|---|---|---|
| `max_seq_length` | reduce | less context per example |
| batch size | reduce | noisier gradients, slower training |
| gradient accumulation | increase while batch is small | more steps before optimizer update |
| LoRA rank | reduce | less adapter capacity |
| target modules | fewer modules | less expressive adaptation |
| gradient checkpointing | enable | slower backward pass |
| model size | choose smaller | lower ceiling quality |

## 10. Common QLoRA Failure Modes

| Symptom | Likely cause | Fix |
|---|---|---|
| `bitsandbytes` import error | incompatible OS/CUDA/package | use supported CUDA environment or container |
| target module not found | wrong module names for model family | inspect `model.named_modules()` |
| CUDA OOM at load time | model too large or context too long | smaller model, lower max sequence length |
| loss decreases but outputs worsen | bad data, too high LR, wrong labels | inspect examples and eval set |
| training is slow | checkpointing, small batch, CPU offload | accept trade-off or use larger GPU |
| adapter saves but cannot reload | missing base model or tokenizer mismatch | record base model ID and adapter version |

## 11. QLoRA Run Card Template

Record these details for every run:

```md
# QLoRA Run Card

Base model:
Dataset version:
Train examples:
Eval examples:
Max sequence length:
Quantization: NF4, double quantization, compute dtype
LoRA rank / alpha / dropout:
Target modules:
Batch size / gradient accumulation:
Learning rate:
Hardware:
Main eval result:
Known risks:
```

## Summary

QLoRA is LoRA on top of a 4-bit frozen base model. It is mainly a memory strategy: keep the large model cheap to load, then train small adapters. The core workflow is: load quantized base, prepare for k-bit training, attach LoRA, train adapters, evaluate carefully, then save adapter artifacts.

Next, we will evaluate fine-tuned adapters, merge them when needed, and prepare them for serving.